In [ ]:
import os, subprocess
REPO = 'Damaten'
env = dict(os.environ, GIT_LFS_SKIP_SMUDGE='1')
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/Koushien552/Damaten.git'], check=True, env=env)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], env=env)
subprocess.run(['git', '-C', REPO, 'lfs', 'install'], check=True)
subprocess.run(['git', '-C', REPO, 'lfs', 'pull', '--include', 'models/hex_model.nn'], check=True)
hdr = open(os.path.join(REPO, 'models', 'hex_model.nn'), 'r', errors='replace').readline().strip()
print('model:', hdr)
assert hdr.startswith('HEX')

In [ ]:
import os, subprocess
def build(extra):
    return subprocess.run(['g++', '-O3', '-std=c++17'] + extra +
                          ['-o', 'Damaten/hexai', 'Damaten/src/main.cpp'],
                          capture_output=True, text=True)
r = build(['-mavx2', '-mfma'])
if r.returncode != 0:
    r = build([])
print('hexai build OK' if r.returncode == 0 and os.path.exists('Damaten/hexai') else r.stderr[-2000:])

In [ ]:
import os, subprocess
subprocess.run(['apt-get', 'install', '-y', '-qq', 'libboost-all-dev', 'libdb-dev', 'cmake'],
               stdout=subprocess.DEVNULL, check=True)

MOHEX_DIR = 'benzene-vanilla-cmake'
if not os.path.isdir(MOHEX_DIR):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/cgao3/benzene-vanilla-cmake.git'], check=True)
os.makedirs(os.path.join(MOHEX_DIR, 'build'), exist_ok=True)

r = subprocess.run(['cmake', '..'], cwd=os.path.join(MOHEX_DIR, 'build'), capture_output=True, text=True)
if r.returncode == 0:
    r = subprocess.run(['make', '-j4'], cwd=os.path.join(MOHEX_DIR, 'build'), capture_output=True, text=True)

MOHEX = os.path.join(MOHEX_DIR, 'build', 'src', 'mohex', 'mohex')
print('mohex build OK' if r.returncode == 0 and os.path.exists(MOHEX) else (r.stdout[-1500:] + r.stderr[-1500:]))

In [ ]:
# 1局だけ動作確認（HexAI=黒）。設定がおかしくないかをすぐ確認できる。
!python Damaten/colab/bench_vs_mohex.py \
  --mohex benzene-vanilla-cmake/build/src/mohex/mohex \
  --hexai Damaten/hexai --model Damaten/models/hex_model.nn \
  --hexai-iters 1500 --verify-only

In [ ]:
HEXAI_ITERS = 1500          # HexAIの強さ（MCTSシミュレーション数、固定）
BUDGETS = '0.1,0.5,2.0'    # MoHexの思考時間スイープ（秒/手）
COARSE_GAMES = 20           # 各budgetでの粗い対局数
FINAL_GAMES = 100           # 50%に最も近いbudgetでの本番対局数
OPENINGS = 4                # ランダムオープニングの手数（色を入れ替えて先手有利を相殺）

!python Damaten/colab/bench_vs_mohex.py \
  --mohex benzene-vanilla-cmake/build/src/mohex/mohex \
  --hexai Damaten/hexai --model Damaten/models/hex_model.nn \
  --hexai-iters {HEXAI_ITERS} --budgets {BUDGETS} \
  --coarse-games {COARSE_GAMES} --final-games {FINAL_GAMES} --openings {OPENINGS} \
  --out bench_results.csv